# Medical Triage — GRPO Fine-tuning

Fine-tunes **Qwen2.5-1.5B-Instruct** using **Group Relative Policy Optimization (GRPO)**  
on the [Medical Triage Environment](https://huggingface.co/spaces/kunalkachru23/medical-triage-env).

The live HF Space acts as the reward oracle — no local server needed.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Cell 1 — Install dependencies
!pip install trl>=0.12.0 transformers>=4.40.0 accelerate bitsandbytes peft datasets requests -q

In [ ]:
# Cell 1b — Mount Google Drive (checkpoints + adapter survive runtime disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/grpo-medical-triage'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Checkpoints will save to: {DRIVE_DIR}')

In [ ]:
# Cell 2 — Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 3 — Config
SERVER_URL        = 'https://kunalkachru23-medical-triage-env.hf.space'
MODEL_ID          = 'Qwen/Qwen2.5-1.5B-Instruct'
HF_TOKEN          = ''  # optional — only needed to push adapter to Hub
OUTPUT_DIR        = DRIVE_DIR  # saves to Google Drive — survives disconnects
PROMPTS_PER_TASK  = 8   # 8 x 11 tasks = 88 prompts total
NUM_GENERATIONS   = 4   # GRPO group size G
EPOCHS            = 2   # 2 epochs on 88 prompts = ~176 optimizer steps

TASKS = [
    'simple_triage', 'conflicting_vitals', 'masked_deterioration',
    'demographic_fairness', 'deteriorating_patient', 'sepsis_bundle',
    'paediatric_triage', 'medication_reconciliation',
    'icu_deterioration', 'sbar_handover', 'differential_diagnosis',
]

In [ ]:
# Cell 4 — Health check
import requests
r = requests.get(f'{SERVER_URL}/health', timeout=15)
print('Server health:', r.json())

In [ ]:
# Cell 5 — Build dataset from live environment
from datasets import Dataset

records = []
for task in TASKS:
    count = 0
    for _ in range(PROMPTS_PER_TASK):
        try:
            r = requests.post(f'{SERVER_URL}/reset', json={'task_id': task}, timeout=30)
            r.raise_for_status()
            obs = r.json()['observation']
            patient_text = obs.get('patient_history') or obs.get('patient_presentation', '')
            # task_description removed — it uses different priority values and conflicts with system prompt
            prompt = f'[TASK: {task}]\n\n{patient_text}'.strip()
            records.append({'prompt': prompt, 'task': task})
            count += 1
        except Exception as e:
            print(f'Warning: /reset failed ({task}): {e}')
    print(f'  {task}: {count}/{PROMPTS_PER_TASK} prompts')

dataset = Dataset.from_list(records)
print(f'\nTotal: {len(dataset)} prompts across {len(TASKS)} tasks')
print('Sample:', dataset[0]['prompt'][:200])

In [ ]:
# Cell 6 — Load tokenizer + apply chat template
from transformers import AutoTokenizer

SYSTEM_PROMPT = (
    'You are a clinical triage AI assistant.\n'
    'The user message begins with [TASK: <name>]. Respond with ONE single flat JSON object for that task only.\n'
    'No nested objects. No markdown. No code fences. No extra text.\n\n'
    'If [TASK: simple_triage] or [TASK: conflicting_vitals] or [TASK: masked_deterioration] or [TASK: demographic_fairness]:\n'
    '{"priority":"low|medium|high|critical","news2_score":<integer 0-20>,'
    '"critical_sign":"<most abnormal vital sign>","recommended_action":"emergency_response|urgent_review|routine_monitoring",'
    '"rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: deteriorating_patient]:\n'
    '{"action":"monitor|escalate|emergency_response|comfort_care","rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: sepsis_bundle]:\n'
    '{"priority":"low|medium|high|critical",'
    '"bundle_elements":["blood_cultures","iv_antibiotics","iv_fluids","lactate","vasopressors"],'
    '"antibiotic_choice":"<str>","fluid_volume_ml":<integer>,"vasopressor_indicated":<true|false>,'
    '"rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: paediatric_triage]:\n'
    '{"priority":"low|medium|high|critical",'
    '"age_group":"infant|toddler|preschool|school_age|adolescent","pews_score":<integer 0-13>,'
    '"critical_sign":"<most abnormal vital sign>","recommended_action":"emergency_response|urgent_review|routine_monitoring",'
    '"rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: medication_reconciliation]:\n'
    '{"issues_found":["<str>"],"severity":"critical|high|moderate|low","requires_pharmacist":<true|false>,'
    '"recommended_action":"safe_to_prescribe|modify_dose|withhold_drug|emergency_review",'
    '"drug_to_withhold":"<drug name or none>","rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: icu_deterioration]:\n'
    '{"sofa_score":<integer 0-24>,"primary_organ_failure":"cardiovascular|respiratory|renal|hepatic|neurological|coagulation",'
    '"deterioration_trend":"improving|stable|worsening",'
    '"intervention":"maintain_current|increase_support|emergency_escalation|prepare_palliation",'
    '"rationale":"<str>"}\n\n'
    'If [TASK: sbar_handover]:\n'
    '{"escalation_required":<true|false>,"priority":"low|medium|high|critical",'
    '"assessment":"<string describing clinical situation>",'
    '"recommendation":"routine_monitoring|urgent_review|emergency_response"}\n\n'
    'If [TASK: differential_diagnosis]:\n'
    '{"must_not_miss":"<life-threatening diagnosis to exclude>","top_diagnosis":"<most likely diagnosis>",'
    '"differentials":["<str>","<str>","<str>"],"first_investigation":"<most important first test>",'
    '"urgency":"immediate|urgent|routine"}'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

def format_prompt(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': example['prompt']},
    ]
    example['prompt'] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return example

dataset = dataset.map(format_prompt)
print('Tokenizer loaded.')
print('Sample (first 300 chars):', dataset[0]['prompt'][:300])

In [ ]:
# Cell 7 — Load model with 4-bit quantization
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

used_gb  = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded. VRAM: {used_gb:.1f} GB used / {total_gb:.0f} GB total')

In [ ]:
# Cell 8 — Apply LoRA adapter
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 9 — Reward function (FIXED: parse JSON fields directly, not {"response":"..."})
import uuid, time, re, json as _json
from requests.adapters import HTTPAdapter, Retry

REQUEST_TIMEOUT = 25
FALLBACK_REWARD = 0.0001
DEBUG_LOGGING = True

_session = requests.Session()
try:
    _retry = Retry(total=3, backoff_factor=0.7,
                   status_forcelist=[429, 500, 502, 503, 504],
                   allowed_methods=['POST'], raise_on_status=False)
except TypeError:
    _retry = Retry(total=3, backoff_factor=0.7,
                   status_forcelist=[429, 500, 502, 503, 504],
                   method_whitelist=['POST'], raise_on_status=False)
_session.mount('https://', HTTPAdapter(max_retries=_retry))

def reward_fn(completions, prompts=None, task=None, **kwargs):
    batch_size = len(completions)
    task_ids = [task] * batch_size if isinstance(task, str) else (task or ['simple_triage'] * batch_size)
    rewards, success_count = [], 0
    start = time.perf_counter()

    for idx, (completion, task_id) in enumerate(zip(completions, task_ids)):
        # Strip markdown fences then parse JSON into a dict
        completion = re.sub(r'```(?:json)?\s*|\s*```', '', completion).strip()
        try:
            action_dict = _json.loads(completion)
        except Exception:
            action_dict = {}

        try:
            sid = str(uuid.uuid4())
            r1 = _session.post(f'{SERVER_URL}/reset',
                json={'task_id': task_id, 'session_id': sid}, timeout=REQUEST_TIMEOUT)
            r1.raise_for_status()
            # KEY FIX: send parsed fields directly as action — NOT {"response": "..."}
            r2 = _session.post(f'{SERVER_URL}/step',
                json={'session_id': sid, 'action': action_dict}, timeout=REQUEST_TIMEOUT)
            r2.raise_for_status()
            reward = float(r2.json()['reward'])
            success_count += 1
            if DEBUG_LOGGING:
                print(f'  [{task_id}] reward={reward:.4f} ({idx+1}/{batch_size})')
        except Exception as e:
            reward = FALLBACK_REWARD
            if DEBUG_LOGGING:
                print(f'  [error] {task_id}: {str(e)[:60]}')
        rewards.append(reward)

    elapsed = time.perf_counter() - start
    print(f'\nBatch: {success_count}/{batch_size} OK | {elapsed:.2f}s\n')
    return rewards

# Smoke test using paired session (same case for generate + grade)
print('Running smoke test...')
_sid = str(uuid.uuid4())
_r1 = requests.post(f'{SERVER_URL}/reset', json={'task_id': 'simple_triage', 'session_id': _sid}, timeout=30)
_patient = _r1.json()['observation'].get('patient_history', '')
print('Case:', _patient[:100])
_action = {'priority': 'high', 'news2_score': 7, 'critical_sign': 'spo2',
           'recommended_action': 'emergency_response', 'rationale': 'test', 'confidence': 0.9}
_r2 = requests.post(f'{SERVER_URL}/step',
    json={'session_id': _sid, 'action': _action}, timeout=30)
print('Smoke test reward:', _r2.json()['reward'])
print('(Any value > 0.0001 confirms the API pipeline is working correctly)')

In [ ]:
# Cell 10 — GRPO training (TRL 1.1.0 compatible)
import trl
print(f"TRL Version: {trl.__version__}")
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=300,
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    logging_steps=1,
    save_steps=10,
    save_total_limit=3,
    fp16=False,
    bf16=True,
    report_to='none',
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=[reward_fn],
)

print(f'Training: {len(dataset)} prompts x {EPOCHS} epoch(s), G={NUM_GENERATIONS}')
print(f'Checkpoints saving to: {OUTPUT_DIR} every 10 steps')
trainer.train()

In [ ]:
# Cell 11 — Save final adapter to Google Drive
import os
save_path = os.path.join(OUTPUT_DIR, 'final')
os.makedirs(save_path, exist_ok=True)
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f'Adapter saved to: {save_path}')
print('Files:', os.listdir(save_path))

In [ ]:
# Cell 12 — (Optional) Push adapter to HF Hub
hub_repo_id = 'kunalkachru23/grpo-medical-triage-qwen1.5b'

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(hub_repo_id)
    tokenizer.push_to_hub(hub_repo_id)
    print(f'Pushed to: https://huggingface.co/{hub_repo_id}')
else:
    print('HF_TOKEN not set — skipping. Set HF_TOKEN in Cell 3 to enable.')